# M2 v2 · Selección en validación

**TFM · HAM10000** · Responsable: **Alex Reinoso**. Sin GPU. No hace falta Drive ni el dataset.

Lee los 9 runs `estado=final` de `m2-entrenamiento-v2`, aplica la misma regla de la v1 y publica `m2-seleccion-v2`. Mueve el alias `final` a la semilla representativa de cada red y `seleccionado` a la ganadora. Los alias dejan de apuntar a la v1.

Después guarda `umbral_mel` en esos tres runs. Son los umbrales ya medidos en validación. No abre el conjunto de prueba.

Secretos `DAGSHUB_USER` y `DAGSHUB_TOKEN`. Si `m2-seleccion-v2` ya está en `estado=final`, la segunda celda se niega a repetirlo.


In [ ]:
OWNER, REPO, BRANCH = "aReinoso007", "tfm-melanoma-cnn-gradcam", "m2/seleccion-v2"
AUTOR = "alex"
UMBRALES = {
    "c009467e38f04b22a0208f6732c58e1a": 0.127295,  # ResNet50, semilla 2024
    "a16c233b18b043f49a84e5afed0cf3ab": 0.083365,  # DenseNet121, semilla 2024
    "de11d3148353403ba1c091e6b66f0abf": 0.198781,  # EfficientNetB0, semilla 2024
}


In [ ]:
import os
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin
!git checkout -q {BRANCH}
!git pull -q origin {BRANCH}
!git log --oneline -1
!pip install -q -r requirements.txt


In [ ]:
import os
from google.colab import userdata
os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
from src.tracking import load_config, setup_mlflow
from src.selection import lanzar_seleccion_v2
cfg = load_config("configs/base.yaml")
print(setup_mlflow(cfg))
resultado = lanzar_seleccion_v2(cfg, autor=AUTOR)
print(resultado["markdown"])
print("Run de la selección:", resultado["run_id_seleccion"])
print("Ganadora:", resultado["decision"]["ganadora"])


In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
for run_id, t in UMBRALES.items():
    client.log_param(run_id, "umbral_mel", f"{t:.6f}")
    print(run_id, client.get_run(run_id).data.params["umbral_mel"])
